In [402]:
import pandas as pd

In [403]:
data = pd.read_csv('train.csv')

In [404]:
data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [405]:
data.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [406]:
data['Survived'].dtype

dtype('int64')

In [407]:
data['Survived'].value_counts() #небольшой дисбаланс классов

Survived
0    549
1    342
Name: count, dtype: int64

Как мы видим, целевая переменная – дискретная (принимает значения 0/1), что говорит о задачи классификации. Можно использовать accuaracy т.к. ддсиабаланс несильный, но лучше использовать f1-score + accuracy. 

Сначала надо очистить данные и убрать лишние столбцы

In [408]:
data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [409]:
data = data.drop(['Cabin', 'Fare', 'Ticket', 'Name', 'PassengerId', 'Embarked'], axis=1)

In [410]:
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch
0,0,3,male,22.0,1,0
1,1,1,female,38.0,1,0
2,1,3,female,26.0,0,0
3,1,1,female,35.0,1,0
4,0,3,male,35.0,0,0


Заменим пропущенные значения возраста средним значением 

In [411]:
data['Age'].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

In [412]:
data['Age'].isna().sum()

np.int64(177)

In [413]:
data['Age'] = data['Age'].fillna(data.Age.mean())

In [414]:
data['Age'].describe()

count    891.000000
mean      29.699118
std       13.002015
min        0.420000
25%       22.000000
50%       29.699118
75%       35.000000
max       80.000000
Name: Age, dtype: float64

In [415]:
data['Age'].isna().sum()

np.int64(0)

In [416]:
data.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
dtype: int64

In [417]:
data['Family'] = data['SibSp'] + data['Parch']

In [418]:
data['Family'].value_counts()

Family
0     537
1     161
2     102
3      29
5      22
4      15
6      12
10      7
7       6
Name: count, dtype: int64

In [419]:
ohe = pd.get_dummies(data['Sex'], drop_first=True)
ohe.head()

,male
0,True
1,False
2,False
3,False
4,True


In [420]:
data = pd.concat([data, ohe], axis=1).drop(columns=['Sex'])
data.head()

,Survived,Pclass,Age,SibSp,Parch,Family,male
0,0,3,22.0,1,0,1,True
1,1,1,38.0,1,0,1,False
2,1,3,26.0,0,0,0,False
3,1,1,35.0,1,0,1,False
4,0,3,35.0,0,0,0,True


In [421]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X = data.drop('Survived', axis=1)
y = data['Survived']
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=1616)

In [422]:
x_train.head()

,Pclass,Age,SibSp,Parch,Family,male
499,3,24.000000,0,0,0,True
254,3,41.000000,0,2,2,False
333,3,16.000000,2,0,2,True
174,1,56.000000,0,0,0,True
17,2,29.699118,0,0,0,True


In [423]:
y_train.head()

499    0
254    0
333    0
174    0
17     1
Name: Survived, dtype: int64

In [424]:
scaler = StandardScaler()
x_train_scaled = pd.DataFrame(
    scaler.fit_transform(x_train),
    columns=x_train.columns
)
x_test_scaled = pd.DataFrame(
    scaler.transform(x_test),
    columns=x_test.columns
)

In [425]:
x_train_scaled.head()

,Pclass,Age,SibSp,Parch,Family,male
0,0.848058,-0.431913,-0.469444,-0.480626,-0.562506,0.739119
1,0.848058,0.914134,-0.469444,1.951635,0.655497,-1.352963
2,0.848058,-1.065346,1.307261,-0.480626,0.655497,0.739119
3,-1.542249,2.101822,-0.469444,-0.480626,-0.562506,0.739119
4,-0.347096,0.019339,-0.469444,-0.480626,-0.562506,0.739119


Baseline

In [426]:
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(x_train_scaled, y_train)

DummyClassifier(strategy='most_frequent')

In [427]:
log_reg = LogisticRegression()
log_reg.fit(x_train_scaled, y_train)

LogisticRegression()

In [428]:
from sklearn.metrics import accuracy_score

In [429]:
y_pred_dimmy = dummy_clf.predict(x_test_scaled)
print(accuracy_score(y_test, y_pred_dimmy))

0.5964125560538116


In [430]:
y_pred_logreg = log_reg.predict(x_test_scaled)
print(accuracy_score(y_test, y_pred_logreg)) 

0.8116591928251121


In [431]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

In [432]:
confusion_matrix(y_test, y_pred_logreg)

array([[118,  15],
       [ 27,  63]])

In [433]:
precision_score(y_test, y_pred_logreg)

0.8076923076923077

In [434]:
recall_score(y_test, y_pred_logreg)

0.7

In [436]:
from sklearn.metrics import f1_score
f1_score(y_test, y_pred_logreg)

0.75

In [437]:
f1_score(y_test, y_pred_dimmy)

0.0